<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>01. Hamiltonian Monte Carlo (HMC)</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [From Random Walk to Hamiltonian Dynamics](#-part-i-from-random-walk-to-hamiltonian-dynamics)**
    - 1.1 [The Problem with Random Walk MCMC](#the-problem-with-random-walk-mcmc)
    - 1.2 [Hamiltonian Mechanics Intuition](#hamiltonian-mechanics-intuition)

- **2. [Setup & Imports](#-part-ii-setup--imports)**

- **3. [HMC Algorithm Deep Dive](#-part-iii-hmc-algorithm-deep-dive)**
    - 3.1 [Leapfrog Integration](#leapfrog-integration)
    - 3.2 [HMC from Scratch](#hmc-from-scratch)
    - 3.3 [Comparing HMC vs Random Walk](#comparing-hmc-vs-random-walk)

- **4. [HMC with TensorFlow Probability](#-part-iv-hmc-with-tensorflow-probability)**
    - 4.1 [Basic HMC Sampling](#basic-hmc-sampling)
    - 4.2 [Adaptive Step Size with Dual Averaging](#adaptive-step-size)

- **5. [NUTS: The No-U-Turn Sampler](#-part-v-nuts-the-no-u-turn-sampler)**
    - 5.1 [Why NUTS?](#why-nuts)
    - 5.2 [NUTS with TFP](#nuts-with-tfp)

- **6. [Practical Example: Bayesian Neural Network Inference](#-part-vi-practical-example)**

- **7. [Key Takeaways](#-part-vii-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. From Random Walk to Hamiltonian Dynamics</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.1. The Problem with Random Walk MCMC</span>

- **Random Walk Metropolis** explores the parameter space by taking small random steps
- In **high dimensions**, this becomes extremely inefficient — the chain takes a long time to explore the posterior
- **HMC solves this** by using gradient information to make large, directed moves while maintaining high acceptance rates

| **Property** | **Random Walk MH** | **HMC** | **NUTS** |
|-------------|-------------------|---------|----------|
| Uses gradients | ❌ No | ✅ Yes | ✅ Yes |
| Scales to high-D | ❌ Poorly | ✅ Well | ✅ Well |
| Step size tuning | Manual | Manual | **Auto** |
| Path length tuning | N/A | Manual | **Auto** |
| Typical acceptance | ~23% (optimal) | ~65% | ~65% |

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.2. Hamiltonian Mechanics Intuition</span>

- Imagine a **frictionless puck** sliding on a surface shaped like the negative log posterior
- The puck has **position** $q$ (= parameter values) and **momentum** $p$ (= auxiliary variables)
- The **Hamiltonian** is the total energy:

$$H(q, p) = U(q) + K(p) = -\log p(q | \mathcal{D}) + \frac{1}{2} p^T M^{-1} p$$

- Where $U(q) = -\log p(q | \mathcal{D})$ is the **potential energy** and $K(p)$ is the **kinetic energy**
- Hamilton's equations describe the dynamics:

$$\frac{dq}{dt} = \frac{\partial H}{\partial p} = M^{-1}p, \quad \frac{dp}{dt} = -\frac{\partial H}{\partial q} = -\nabla U(q)$$

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import time

tfd = tfp.distributions
tfb = tfp.bijectors

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow version: {tf.__version__}")
print(f"TensorFlow Probability version: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. HMC Algorithm Deep Dive</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Leapfrog Integration</span>

The **leapfrog integrator** numerically simulates Hamiltonian dynamics while preserving volume (symplecticity):

1. Half-step momentum update: $p_{t+\epsilon/2} = p_t - \frac{\epsilon}{2} \nabla U(q_t)$
2. Full-step position update: $q_{t+\epsilon} = q_t + \epsilon \cdot M^{-1} p_{t+\epsilon/2}$
3. Half-step momentum update: $p_{t+\epsilon} = p_{t+\epsilon/2} - \frac{\epsilon}{2} \nabla U(q_{t+\epsilon})$

In [ ]:
def leapfrog(q, p, grad_U, step_size, num_steps):
    """Leapfrog integrator for Hamiltonian dynamics."""
    q = tf.identity(q)
    p = tf.identity(p)
    
    # Half step for momentum
    p = p - 0.5 * step_size * grad_U(q)
    
    for i in range(num_steps - 1):
        # Full step for position
        q = q + step_size * p
        # Full step for momentum
        p = p - step_size * grad_U(q)
    
    # Final full step for position
    q = q + step_size * p
    # Final half step for momentum
    p = p - 0.5 * step_size * grad_U(q)
    
    return q, -p  # Negate momentum for reversibility

print("✅ Leapfrog integrator defined.")
print("The leapfrog method preserves the symplectic structure of Hamiltonian dynamics,")
print("ensuring time-reversibility and volume preservation.")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. HMC from Scratch</span>

In [ ]:
def hmc_sampler(target_log_prob_fn, initial_state, num_samples,
                step_size=0.1, num_leapfrog_steps=10):
    """
    Hamiltonian Monte Carlo sampler from scratch.
    
    Args:
        target_log_prob_fn: Unnormalized log posterior
        initial_state: Starting position (tf.Tensor)
        num_samples: Number of samples to draw
        step_size: Leapfrog step size (epsilon)
        num_leapfrog_steps: Number of leapfrog steps (L)
    """
    dim = initial_state.shape[0]
    samples = []
    current_q = tf.Variable(initial_state, dtype=tf.float32)
    accepted = 0
    
    def grad_U(q):
        with tf.GradientTape() as tape:
            tape.watch(q)
            U = -target_log_prob_fn(q)
        return tape.gradient(U, q)
    
    for i in range(num_samples):
        # Sample momentum
        current_p = tf.random.normal(shape=[dim])
        
        # Leapfrog integration
        proposed_q, proposed_p = leapfrog(
            tf.identity(current_q), current_p,
            grad_U, step_size, num_leapfrog_steps
        )
        
        # Compute Hamiltonian
        current_H = -target_log_prob_fn(current_q) + 0.5 * tf.reduce_sum(current_p**2)
        proposed_H = -target_log_prob_fn(proposed_q) + 0.5 * tf.reduce_sum(proposed_p**2)
        
        # Accept/reject
        log_accept = current_H - proposed_H
        if tf.math.log(tf.random.uniform([])) < log_accept:
            current_q.assign(proposed_q)
            accepted += 1
        
        samples.append(current_q.numpy().copy())
    
    return np.array(samples), accepted / num_samples


# Target: 2D correlated Gaussian
target = tfd.MultivariateNormalFullCovariance(
    loc=[0., 0.],
    covariance_matrix=[[1.0, 0.9], [0.9, 1.0]]
)

hmc_samples, hmc_acc = hmc_sampler(
    target_log_prob_fn=lambda q: target.log_prob(q),
    initial_state=tf.constant([0., 0.]),
    num_samples=5000,
    step_size=0.15,
    num_leapfrog_steps=20
)

print(f"HMC acceptance rate: {hmc_acc:.3f}")
print(f"Sample mean: {hmc_samples[500:].mean(axis=0)}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.3. Comparing HMC vs Random Walk</span>

In [ ]:
# Run Random Walk for comparison
rwm_kernel = tfp.mcmc.RandomWalkMetropolis(
    target_log_prob_fn=target.log_prob,
    new_state_fn=tfp.mcmc.random_walk_normal_fn(scale=0.3)
)

@tf.function
def run_rw():
    return tfp.mcmc.sample_chain(
        num_results=5000, num_burnin_steps=500,
        current_state=[0., 0.], kernel=rwm_kernel,
        trace_fn=lambda _, pkr: pkr.is_accepted
    )

rw_samples, rw_accepted = run_rw()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Random Walk samples
axes[0].scatter(rw_samples[:, 0], rw_samples[:, 1], alpha=0.15, s=8, c='#ef4444')
axes[0].plot(rw_samples[:200, 0], rw_samples[:200, 1], 'k-', alpha=0.3, linewidth=0.5)
axes[0].set_title(f'Random Walk Metropolis\nAcceptance: {tf.reduce_mean(tf.cast(rw_accepted, tf.float32)):.1%}',
                  fontsize=15, fontweight='bold')
axes[0].set_xlabel('$x_1$', fontsize=13)
axes[0].set_ylabel('$x_2$', fontsize=13)
axes[0].set_xlim(-4, 4); axes[0].set_ylim(-4, 4)

# HMC samples
axes[1].scatter(hmc_samples[500:, 0], hmc_samples[500:, 1], alpha=0.15, s=8, c='#3b82f6')
axes[1].plot(hmc_samples[:200, 0], hmc_samples[:200, 1], 'k-', alpha=0.3, linewidth=0.5)
axes[1].set_title(f'Hamiltonian Monte Carlo\nAcceptance: {hmc_acc:.1%}',
                  fontsize=15, fontweight='bold')
axes[1].set_xlabel('$x_1$', fontsize=13)
axes[1].set_ylabel('$x_2$', fontsize=13)
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4)

plt.suptitle('Random Walk vs HMC: Sampling a 2D Correlated Gaussian', 
             fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. HMC with TensorFlow Probability</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. Basic HMC Sampling</span>

In [ ]:
# Define a more complex target: banana-shaped distribution
def banana_log_prob(x):
    """Log probability of a banana-shaped distribution."""
    return -0.5 * (x[0]**2 / 10.0 + (x[1] - x[0]**2)**2)

# HMC with TFP
hmc_kernel = tfp.mcmc.HamiltonianMonteCarlo(
    target_log_prob_fn=banana_log_prob,
    step_size=0.3,
    num_leapfrog_steps=20
)

@tf.function
def run_hmc():
    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=5000,
        num_burnin_steps=1000,
        current_state=[0.0, 0.0],
        kernel=hmc_kernel,
        trace_fn=lambda _, pkr: pkr.is_accepted
    )
    return samples, kernel_results

banana_samples, banana_accepted = run_hmc()

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(banana_samples[:, 0], banana_samples[:, 1], alpha=0.2, s=6, c='#8b5cf6')
ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('HMC Sampling: Banana-Shaped Distribution\n'
             f'Acceptance rate: {tf.reduce_mean(tf.cast(banana_accepted, tf.float32)):.1%}',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.2. Adaptive Step Size with Dual Averaging</span>

In [ ]:
# Use SimpleStepSizeAdaptation to automatically tune step size
adaptive_hmc = tfp.mcmc.SimpleStepSizeAdaptation(
    inner_kernel=tfp.mcmc.HamiltonianMonteCarlo(
        target_log_prob_fn=banana_log_prob,
        step_size=1.0,  # Initial step size (will be adapted)
        num_leapfrog_steps=20
    ),
    num_adaptation_steps=int(0.8 * 1000),  # Adapt during 80% of burn-in
    target_accept_prob=0.75  # Target acceptance rate
)

@tf.function
def run_adaptive_hmc():
    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=5000,
        num_burnin_steps=1000,
        current_state=[0.0, 0.0],
        kernel=adaptive_hmc,
        trace_fn=lambda _, pkr: {
            'is_accepted': pkr.inner_results.is_accepted,
            'step_size': pkr.new_step_size
        }
    )
    return samples, kernel_results

adaptive_samples, adaptive_results = run_adaptive_hmc()

acc_rate = tf.reduce_mean(tf.cast(adaptive_results['is_accepted'], tf.float32))
final_step = adaptive_results['step_size'][-1]

print(f"Adapted step size: {final_step:.4f}")
print(f"Final acceptance rate: {acc_rate:.3f}")
print(f"ESS (dim 0): {tfp.mcmc.effective_sample_size(adaptive_samples[:, 0]).numpy():.1f}")
print(f"ESS (dim 1): {tfp.mcmc.effective_sample_size(adaptive_samples[:, 1]).numpy():.1f}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. NUTS: The No-U-Turn Sampler</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">5.1. Why NUTS?</span>

- Standard HMC requires choosing **two hyperparameters**: step size $\epsilon$ and number of leapfrog steps $L$
- **NUTS automatically determines $L$** by detecting when the trajectory starts to turn back ("U-turn")
- This makes NUTS much more user-friendly — you only need to adapt the step size

> **In practice, NUTS is almost always preferred over standard HMC.**

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">5.2. NUTS with TFP</span>

In [ ]:
# NUTS with adaptive step size
nuts_kernel = tfp.mcmc.SimpleStepSizeAdaptation(
    inner_kernel=tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=banana_log_prob,
        step_size=0.5,
        max_tree_depth=10
    ),
    num_adaptation_steps=800,
    target_accept_prob=0.80
)

@tf.function
def run_nuts():
    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=5000,
        num_burnin_steps=1000,
        current_state=[0.0, 0.0],
        kernel=nuts_kernel,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted
    )
    return samples, kernel_results

nuts_samples, nuts_accepted = run_nuts()

print(f"NUTS acceptance rate: {tf.reduce_mean(tf.cast(nuts_accepted, tf.float32)):.3f}")
print(f"NUTS ESS (dim 0): {tfp.mcmc.effective_sample_size(nuts_samples[:, 0]).numpy():.1f}")
print(f"NUTS ESS (dim 1): {tfp.mcmc.effective_sample_size(nuts_samples[:, 1]).numpy():.1f}")

In [ ]:
# Compare all three methods side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, samps, title, color in zip(
    axes,
    [banana_samples, adaptive_samples, nuts_samples],
    ['Standard HMC', 'Adaptive HMC', 'NUTS'],
    ['#ef4444', '#3b82f6', '#22c55e']
):
    ess0 = tfp.mcmc.effective_sample_size(samps[:, 0]).numpy()
    ess1 = tfp.mcmc.effective_sample_size(samps[:, 1]).numpy()
    ax.scatter(samps[:, 0], samps[:, 1], alpha=0.15, s=6, c=color)
    ax.set_title(f'{title}\nESS: ({ess0:.0f}, {ess1:.0f})',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('$x_1$', fontsize=13)
    ax.set_ylabel('$x_2$', fontsize=13)
    ax.set_xlim(-8, 8); ax.set_ylim(-5, 25)

plt.suptitle('Comparison of HMC Variants on Banana Distribution', 
             fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Practical Example: Bayesian Neural Network Inference</span>

In [ ]:
# Generate nonlinear regression data
np.random.seed(42)
n = 100
X_train = np.random.uniform(-3, 3, n).astype(np.float32)
y_train = (np.sin(X_train) + 0.3 * np.random.randn(n)).astype(np.float32)

# Define a simple 1-hidden-layer BNN with HMC
hidden_units = 10
# Total params: (1 * 10 + 10) + (10 * 1 + 1) = 21
num_params = (1 * hidden_units + hidden_units) + (hidden_units * 1 + 1)

def bnn_log_prob(params):
    """Log posterior of a small BNN."""
    # Unpack parameters
    idx = 0
    W1 = tf.reshape(params[idx:idx+hidden_units], [1, hidden_units]); idx += hidden_units
    b1 = params[idx:idx+hidden_units]; idx += hidden_units
    W2 = tf.reshape(params[idx:idx+hidden_units], [hidden_units, 1]); idx += hidden_units
    b2 = params[idx:idx+1]; idx += 1
    
    # Forward pass
    h = tf.nn.tanh(tf.expand_dims(X_train, 1) @ W1 + b1)
    pred = tf.squeeze(h @ W2 + b2)
    
    # Likelihood
    log_likelihood = tf.reduce_sum(tfd.Normal(pred, 0.3).log_prob(y_train))
    
    # Prior: Normal(0, 1) on all weights
    log_prior = tf.reduce_sum(tfd.Normal(0., 1.).log_prob(params))
    
    return log_likelihood + log_prior

# Run HMC
nuts = tfp.mcmc.SimpleStepSizeAdaptation(
    inner_kernel=tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=bnn_log_prob,
        step_size=0.01
    ),
    num_adaptation_steps=400,
    target_accept_prob=0.75
)

@tf.function
def sample_bnn():
    return tfp.mcmc.sample_chain(
        num_results=500,
        num_burnin_steps=500,
        current_state=tf.random.normal([num_params]) * 0.1,
        kernel=nuts,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted
    )

bnn_samples, bnn_accepted = sample_bnn()
print(f"BNN HMC acceptance rate: {tf.reduce_mean(tf.cast(bnn_accepted, tf.float32)):.3f}")
print(f"Number of posterior samples: {bnn_samples.shape[0]}")
print(f"Parameters per sample: {bnn_samples.shape[1]}")

In [ ]:
# Posterior predictive
X_test = np.linspace(-4, 4, 200).astype(np.float32)

def bnn_predict(params, x):
    idx = 0
    W1 = tf.reshape(params[idx:idx+hidden_units], [1, hidden_units]); idx += hidden_units
    b1 = params[idx:idx+hidden_units]; idx += hidden_units
    W2 = tf.reshape(params[idx:idx+hidden_units], [hidden_units, 1]); idx += hidden_units
    b2 = params[idx:idx+1]
    h = tf.nn.tanh(tf.expand_dims(x, 1) @ W1 + b1)
    return tf.squeeze(h @ W2 + b2)

# Get predictions from posterior samples
predictions = []
for i in range(bnn_samples.shape[0]):
    predictions.append(bnn_predict(bnn_samples[i], X_test).numpy())
predictions = np.array(predictions)

mean_pred = predictions.mean(axis=0)
std_pred = predictions.std(axis=0)

fig, ax = plt.subplots(figsize=(14, 7))

# Posterior samples
for i in range(min(50, len(predictions))):
    ax.plot(X_test, predictions[i], color='#8b5cf6', alpha=0.05, linewidth=1)

# Uncertainty bands
ax.fill_between(X_test, mean_pred - 2*std_pred, mean_pred + 2*std_pred,
                alpha=0.2, color='#3b82f6', label='±2σ (95% CI)')
ax.fill_between(X_test, mean_pred - std_pred, mean_pred + std_pred,
                alpha=0.3, color='#3b82f6', label='±1σ (68% CI)')

# Mean and true function
ax.plot(X_test, mean_pred, color='#06b6d4', linewidth=2.5, label='Posterior mean')
ax.plot(X_test, np.sin(X_test), 'r--', linewidth=2, label='True function: sin(x)')
ax.scatter(X_train, y_train, c='#1e293b', s=30, zorder=5, 
           edgecolors='white', linewidth=0.5, label='Training data')

ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', fontsize=14)
ax.set_title('Bayesian Neural Network with HMC Inference\n'
             'Uncertainty grows away from training data', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(-4, 4)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">7. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| HMC | Uses gradient info to make large, efficient moves |
| Leapfrog | Symplectic integrator that preserves volume |
| Step size | Too large → low acceptance; too small → slow exploration |
| NUTS | Automatically tunes path length — **use this in practice** |
| Adaptive HMC | `SimpleStepSizeAdaptation` tunes step size during warmup |
| BNN + HMC | Full posterior over weights → uncertainty quantification |

### 🎯 Practical Recommendations
1. **Always use NUTS** unless you have a specific reason not to
2. **Use adaptive step size** during the burn-in/warmup phase
3. **Run multiple chains** to verify convergence with R-hat
4. **Check ESS** to ensure you have enough effective samples
5. **Target ~65–80% acceptance rate** for HMC/NUTS